In [11]:
using DifferentialEquations     # For solving differential equations
using LinearAlgebra             # Provides linear algebra functionalities
using Plots                     # For plotting results
include("Chemkin.jl")

n_s = 7
n_r = 6

# Initial species concentrations: N2, O2, CH4, H2O, CO2, CO, H2
species_names = ["N2", "O2", "CH4", "H2O", "CO2", "CO", "H2"]
X0 = zeros(length(species_names)+1)
X0[1] = 900 # K
X0[2:end] = 20.06*[7.55/10.57, 2/10.57, 1/10.57, 0.02/10.57, 0, 0, 0] #mol/m^
# NOTE: H2O concentration cannot be zero or the code does not run.

# Atomic accounting check
println("Pre integration atomic accounting:")
println("The amount of hydrogen atoms is " *
        string(X0[4]*4+X0[5]*2+X0[8]*2)* " moles")
println("The amount of carbon atoms is " *
        string(X0[4]+X0[6]+X0[7])* " moles")
println("The amount of oxygen atoms is " *
        string(X0[3]*2+X0[5]+X0[6]*2+X0[7])* " moles")

# Stoichiometric matrix
S = [0 -0.5 -1 0 0 1 2; 
     0 0 -1 -1 0 1 3; 
     0 -0.5 0 1 0 0 -1; 
     0 0.5 0 -1 0 0 1; 
     0 0 0 -1 1 -1 1; 
     0 0 0 1 -1 1 -1]

species_dict = load_chemical_data("CHEMKIN-THERMDAT.txt")

Pre integration atomic accounting:
The amount of hydrogen atoms is 7.66720908230842 moles
The amount of carbon atoms is 1.8978240302743614 moles
The amount of oxygen atoms is 7.629252601702933 moles


Dict{String, Tuple{Float64, Vector{Float64}}} with 550 entries:
  "HSIC"          => (1500.0, [5.84954, 0.000762835, -9.97413e-8, -3.81159e-11,…
  "H2S"           => (1000.0, [2.88315, 0.00382783, -1.4234e-6, 2.498e-10, -1.6…
  "SIF3NHSIH3"    => (1000.0, [16.6994, 0.00778978, -8.11057e-7, -7.6502e-10, 1…
  "CLSI(CH3)2CH2" => (1500.0, [21.151, 0.00801827, -7.92425e-7, -3.29505e-10, 5…
  "GEF2"          => (1000.0, [4.76795, 0.00841094, -1.67642e-5, 1.56225e-8, -5…
  "CSICL3"        => (1500.0, [12.5054, 0.000533922, -2.58861e-7, 6.07531e-11, …
  "H2GAME"        => (600.0, [5.8316, 0.0122287, 3.03367e-7, -3.95694e-9, 1.225…
  "O2-"           => (1000.0, [3.88301, 0.000740787, -2.96178e-7, 5.7243e-11, -…
  "ASALME"        => (600.0, [7.12711, 0.00735786, 2.3008e-8, -2.2264e-9, 6.927…
  "CH2CLCHCL2"    => (1500.0, [16.1874, 0.00304768, -5.0115e-7, -1.5967e-11, 7.…
  "CHCLCCLOH"     => (1500.0, [14.1221, 0.00258376, -4.5769e-7, 5.21568e-12, 3.…
  "H2SI(CH3)CH2"  => (1500.0, [13.8883, 0.007

In [12]:
# Reaction rate function
function Arrhenius(X)
    R = 1.987204258640

    T = X[1]
    Y = X[2:end]
    
    k  = zeros(6)
    k[1] = 0.44e12 * exp(-30000 / (R * T)) #from Jones & Lindstedt 1988
    k[2] = 0.30e9 * exp(-30000 / (R * T)) #from Jones & Lindstedt 1988
    k[3] = 0.25e17 * T^-1 * exp(-40000 / (R * T)) #from Jones & Lindstedt 1988
    k[4] = 0
    k[5] = 5.0e12 * exp(-67300/ (R * T)) #from Graven & Long 1953
    k[6] = 9.5e10 * exp(-57000/ (R * T)) #from Graven & Long 1953

    # Ensure non-negative concentrations
    Y = max.(Y, 0)  # Avoid zero or negative concentrations for stability

    # Reaction rates
    r = zeros(6)
    r[1] = k[1] * Y[3]^0.5 * Y[2]^1.25 #from Jones & Lindstedt 1988
    r[2] = k[2] * Y[3] * Y[4] #from Jones & Lindstedt 1988
    r[3] = k[3] * Y[7]^0.5 * Y[2]^2.25 * Y[4]^-1 #from Jones & Lindstedt 1988
    r[4] = 0
    r[5] = k[5] * Y[6]^0.5 * Y[4] / (1 + 1.2e4 * Y[7]) #from Graven & Long 1953
    r[6] = k[6] * Y[7]^0.5 * Y[5] / (1 + 3.6e3 * Y[6]) #from Graven & Long 1953

    return r
end

# Temperature rate of change due to reaction enthalpy
function dT(X, r)
    h0_vec = zeros(n_r)              # Enthalpy change per reaction (J/mol)
    cp_vec = zeros(n_s)              # Heat capacity per species (J/(mol·K))

    T = X[1]                         # Temperature in K
    concentrations = X[2:end]        # Species concentrations in mol/m³

    # Pre-compute species enthalpies and heat capacities
    h_species = zeros(n_s)     # Enthalpy for each species (J/mol)
    for (species_index, species) in enumerate(species_names)
        cp_vec[species_index] = species_cp(X[1],species_dict[species_names[species_index]])
        h_species[species_index] = h0(X[1],species_dict[species_names[species_index]])
    end

    # Compute h0_vec for reactions
    for reaction_index in 1:n_r
        # Sum over species: stoichiometric coefficient * species enthalpy
        for species_index in 1:n_s
            stoich_coeff = S'[species_index, reaction_index]
            if stoich_coeff != 0.0
                h0_vec[reaction_index] += stoich_coeff * h_species[species_index]
            end
        end
    end

    # Compute the total enthalpy change rate (J/(m³·s))
    dH = sum(r .* h0_vec)

    # Compute the total heat capacity of the mixture (J/(m³·K))
    c_p = sum(concentrations .* cp_vec)

    # Compute temperature rate of change (K/s)
    dT_dt = -dH / c_p

    return dT_dt
end

dT (generic function with 1 method)

In [13]:
# Derivative function for the ODE system
function f!(dX, X, p, t)
    # Compute differentials
    r = Arrhenius(X)
    dX[1] = dT(X,r)
    dX[2:end] = S' * r  # Species concentrations change
end

f! (generic function with 1 method)

In [28]:
# Define the time span for the simulation (in seconds)
tspan = (0.0, 1)

# Set up the ODE problem using the defined derivative function f!
problem = ODEProblem(f!, X0, tspan)

# solve problem 
@time sol = solve(problem, alg_hints=[:stiff], abstol = 1e-6, reltol = 1e-4)

function relaxation_time(sol)
    p = 5 / 100  # convert to fraction
    final = sol.u[end]

    for (i, u) in enumerate(sol.u)
        # relative error per component
        rel_error = abs.((u .- final) ./ (final .+ eps()))  # avoid divide-by-zero
        if abs.((u .- final) < 1e-9;
                
        if all(rel_error .< p)
            return sol.t[i]
        end
    end
    return NaN  # Not within threshold
end


τ = relaxation_time(sol)
println("Relaxation time: $τ seconds")

  0.062858 seconds (1.04 M allocations: 26.050 MiB, 11.67% gc time)
NaN


In [29]:
# Print an atomic accounting check after the simulation
final_state = sol.u[end]
println("Post integration atomic accounting:")
println("The amount of hydrogen atoms is " *
        string(final_state[4]*4 + final_state[5]*2 + final_state[8]*2) * " moles")
println("The amount of carbon atoms is " *
        string(final_state[4] + final_state[6] + final_state[7]) * " moles")
println("The amount of oxygen atoms is " *
        string(final_state[3]*2 + final_state[5] + final_state[6]*2 + final_state[7]) * " moles")

# Extract the solution arrays
t = sol.t
T = sol[1, :]
C = sol[2:end, :]'

println(maximum(T))

# Create a plot for the temperature
plot1 = plot(t, T, xlabel="Time (s)", ylabel="Temperature (K)")

# Create a plot for the species concentrations
plot2 = plot(t, C, xlabel="Time (s)", ylabel="Concentration", label=["N2" "O2" "CH4" "H2O" "CO2" "CO" "H2"])

plot(plot1, plot2, layout = (2,1))

savefig("4step")


Post integration atomic accounting:
The amount of hydrogen atoms is 7.667209082308416 moles
The amount of carbon atoms is 1.897824030274363 moles
The amount of oxygen atoms is 7.629252601702923 moles
2767.641509271989


"C:\\Users\\jelte\\chemicalcombustion\\Toy Models\\4step.png"